In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# === BOOTSTRAP: RUN THIS FIRST ===
import os, sys
from pathlib import Path
REPO_PATH = Path("/content/drive/MyDrive/MaintainAI/code")
os.chdir(REPO_PATH)
sys.path.insert(0, str(REPO_PATH))
print("ROOT:", REPO_PATH)
print("Exists:", REPO_PATH.exists())
print("src exists:", (REPO_PATH / "src").exists())

ROOT: /content/drive/MyDrive/MaintainAI/code
Exists: True
src exists: True


# 06 — Base SLM evaluation (Colab GPU, MANDATORY before fine-tuning)

## Prerequisites
1. Runtime → Change runtime type → GPU (T4)
2. Mount Google Drive for persistence
3. Run this notebook BEFORE notebook 07

## Process
1. Load base model (Qwen2.5-3B-Instruct) via HFBackend (4-bit)
2. Score `data/slm/test.jsonl` (100 held-out scenarios) via `src.slm_eval.run_harness`
3. Save metrics to `reports/metrics_base.json`
4. Paste numbers into `docs/EVALUATION.md`

**Do NOT fine-tune before recording this baseline.**

In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q transformers accelerate bitsandbytes peft 2>&1 | tail -1

# Verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.5 MB/s eta 0:00:00
CUDA: True
GPU: Tesla T4
VRAM: 15.64 GB


In [4]:
# Load base model and run evaluation
import json
from src.slm_service import SLMService, HFBackend
from src.slm_eval import run_harness

# Configure base model (4-bit quantized)
svc = SLMService(
    config={'slm': {'model_name': 'Qwen/Qwen2.5-3B-Instruct'}},
    backend=HFBackend('Qwen/Qwen2.5-3B-Instruct', quantization='4bit')
)

# Load test scenarios (100 decision-point examples from NASA test fleet)
test = [json.loads(l) for l in open('data/slm/test.jsonl')]
print(f'Loaded {len(test)} test scenarios')

def analyze_fn(e):
    raw = svc.backend.generate(e['system'] + '\n' + e['user'])
    from src.slm_service import extract_json
    from src.schemas import SLMAnalysis
    obj = extract_json(raw)
    try:
        SLMAnalysis.model_validate(obj or {})
        return {**obj, 'meta': {'backend': svc.version, 'valid': True}}
    except Exception as ex:
        return {'meta': {'backend': svc.version, 'valid': False, 'reason': str(ex)[:200]}}

# Run harness
m = run_harness(test, analyze_fn)
print(json.dumps(m, indent=1))

# Save metrics
import os
os.makedirs('/content/drive/MyDrive/MaintainAI/reports', exist_ok=True)
with open('/content/drive/MyDrive/MaintainAI/reports/metrics_base.json', 'w') as f:
    json.dump({**m, 'backend': svc.version}, f, indent=1)
print('\n✓ Saved: /content/drive/MyDrive/MaintainAI/reports/metrics_base.json')

Loaded 100 test scenarios


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new

{
 "n": 100,
 "risk_accuracy": 0.0,
 "condition_accuracy": 0.0,
 "evidence_f1_mean": 0.0,
 "recommendation_accuracy": 0.0,
 "validity_rate": 0.0,
 "uncertainty_skill": {
  "unknown_admits_uncertainty": null,
  "known_confident": null,
  "n_unknown": 0
 },
 "latency_mean_ms": 0.0,
 "n_failures": 100
}

✓ Saved: /content/drive/MyDrive/MaintainAI/reports/metrics_base.json


## After running, copy metrics to local repo:
```bash
cp /content/drive/MyDrive/MaintainAI/reports/metrics_base.json reports/
# Then run: python scripts/compare_slm.py
```

Then update `docs/EVALUATION.md` with the measured base model numbers.